# Offline ALNS Repair Model Training — Improved Pipeline

**Colab-ready** | Explicit seeds | Dataset integrity checks | Quality gates | Covariate-shift mitigation

This notebook runs the full offline training pipeline for the Hybrid ALNS repair model.
It is the authoritative, executable version of the improvements documented in `README.md`.

## Pipeline overview

```
generate_dataset (v1) ──► train baseline (v1) ──► collect ALNS states
                                                          │
                         generate_dataset (v2) ──────────┘
                                  │
                         train improved model (v2)  ← quality gate checked here
```

## Improvement areas addressed

| Area | What changed |
|---|---|
| **Reproducibility** | All seeds defined once as constants; passed to every step |
| **Dataset integrity** | Feature version, feature count, label values, NaN checked on load |
| **Dataset summary** | Rows, positive rate, per-source counts, ALNS ratio printed |
| **Quality gates** | ROC-AUC + Average Precision checked on holdout; warns if below threshold |
| **Covariate shift** | ALNS states required for v2+ (`require_alns_states=True`); proportion reported |


## Cell 1 — Colab / local setup

Detects whether the notebook is running on Google Colab or locally.
On Colab, it prompts you to upload the repository zip and installs it.
Locally, it walks up from the current directory to find the repo root.

> **Colab tip**: if this is your first run, upload `bin-packing-optimization.zip`
> when the file picker appears. Subsequent runs can skip the upload if the
> `/content` directory still has the extracted repo.


In [ ]:
import os
import sys
import zipfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules


def find_repo_root(start: str) -> Path:
    """Walk up from `start` until a directory containing both
    pyproject.toml and requirements.txt is found."""
    for root, _, files in os.walk(start):
        if "pyproject.toml" in files and "requirements.txt" in files:
            return Path(root)
    raise FileNotFoundError("Could not find repo root with pyproject.toml")


repo_root = None
if IN_COLAB:
    try:
        repo_root = find_repo_root("/content")
    except FileNotFoundError:
        from google.colab import files

        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("Please upload the repo zip to continue.")
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name, "r") as zip_ref:
            zip_ref.extractall("/content")
        repo_root = find_repo_root("/content")
else:
    repo_root = find_repo_root(os.getcwd())

os.chdir(repo_root)
print("Repo root:", repo_root)


Saving bin-packing-optimization.zip to bin-packing-optimization.zip
Repo root: /content/bin-packing-optimization


## Cell 2 — Install dependencies

Installs Python dependencies from `requirements.txt` and registers the repo
as an editable package so all internal imports resolve correctly.
The `-q` flag suppresses verbose pip output to keep the notebook readable.


In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .


  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for bin-packing-optimization (pyproject.toml) ... done


## Cell 3 — Seed constants and path setup

All randomness is controlled by four constants defined here.
Using separate seeds per step allows independent auditing and partial reruns
without affecting other steps.

| Constant | Used by | Purpose |
|---|---|---|
| `SEED` | `train_repair_model` | Train/test split + GradientBoosting `random_state` |
| `SYNTHETIC_V1_SEED` | `generate_dataset` (v1) | Baseline synthetic dataset |
| `ALNS_SEED` | `collect_alns_states` | ALNS rollout RNG |
| `SYNTHETIC_V2_SEED` | `generate_dataset` (v2) | Supplementary synthetic dataset |


In [ ]:
import random
import numpy as np
import sys
import os
from pathlib import Path

# ── Reproducibility: all seeds defined here, passed to every downstream call ──
SEED              = 42   # train/test split + model random_state
SYNTHETIC_V1_SEED = 0    # generate_dataset v1
ALNS_SEED         = 1    # collect_alns_states
SYNTHETIC_V2_SEED = 2    # generate_dataset v2

random.seed(SEED)
np.random.seed(SEED)

# Ensure the repo root is in sys.path so the package is importable
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

TRAINING_DIR = (
    Path(repo_root)
    / "bin_packing_optimization"
    / "hybrid_learning_metaheuristics"
    / "hybrid_alns"
    / "repair_model_training"
)
DATA_DIR = TRAINING_DIR / "training_data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Change to the training directory so relative paths in config work correctly
os.chdir(TRAINING_DIR)
print("Training dir :", TRAINING_DIR)
print("Data dir     :", DATA_DIR)
print(f"Seeds        : SEED={SEED}, V1={SYNTHETIC_V1_SEED}, ALNS={ALNS_SEED}, V2={SYNTHETIC_V2_SEED}")

# Now imports should work correctly
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.generate_dataset import (
    GenerateDatasetConfig,
    generate_dataset,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.collect_alns_states import (
    CollectAlnsStatesConfig,
    collect_alns_states,
)
from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns.repair_model_training.train_repair_model import (
    TrainRepairModelConfig,
    train_repair_model,
)

Training dir : /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training
Data dir     : /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data
Seeds        : SEED=42, V1=0, ALNS=1, V2=2


## Step 1 — Generate baseline synthetic dataset

Generates `synthetic_v1.pkl`: 4 000 random bin-packing instances with items drawn
from uniform, bimodal, and Gaussian distributions.

For each instance the FFD heuristic builds a start solution; then every feasible
bin placement is labelled using the BFD oracle (1 = minimum-slack bin, 0 = other).

**Expected output** (printed by `generate_dataset`):
- Total rows, positive rate (~0.17 for `max_negatives=5`), class counts.
- An integrity check is run automatically before saving.


In [ ]:
generate_dataset(
    GenerateDatasetConfig(
        instances=4000,
        n_min=50,
        n_max=200,
        max_negatives=5,
        seed=SYNTHETIC_V1_SEED,
        workers=1,          # set > 1 for speed on multi-core Colab runtimes
        output=str(DATA_DIR / "synthetic_v1.pkl"),
    )
)


DATASET GENERATION
  Instances          : 4000
  Instance size range: [50, 200] items
  Max negatives      : 5
  Seed               : 0
  Workers            : 1
  Output             : /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v1.pkl


Generating dataset: 100%|██████████| 4000/4000 [00:10<00:00, 368.85it/s]



Dataset: 707,772 rows x 11 features
  Positive rate : 0.3224  (expected min: 0.1667)
  Class 0 (neg) : 479,551
  Class 1 (pos) : 228,221

Saved: /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v1.pkl  (32.4 MB)
Next step: pass this file to train_repair_model.py via the data config field


{'X': array([[0.49550122, 0.24552146, 0.47244096, ..., 0.20556818, 0.14696765,
         0.9956621 ],
        [0.48975867, 0.23986356, 0.48031497, ..., 0.20556818, 0.14696765,
         0.984123  ],
        [0.4865833 , 0.2367633 , 0.48818898, ..., 0.20556818, 0.14696765,
         0.9777424 ],
        ...,
        [0.11028459, 0.01216269, 0.9910714 , ..., 0.5669997 , 0.5669997 ,
         0.25469863],
        [0.11028459, 0.01216269, 0.9910714 , ..., 0.66267735, 0.66267735,
         0.32694092],
        [0.11028459, 0.01216269, 0.9910714 , ..., 0.57596946, 0.57596946,
         0.26008642]], dtype=float32),
 'y': array([1, 1, 1, ..., 0, 0, 0], dtype=int32),
 'feature_version': 2,
 'source': 'synthetic',
 'summary': {'instances': 4000,
  'n_min': 50,
  'n_max': 200,
  'max_negatives': 5,
  'seed': 0,
  'workers': 1,
  'rows': 707772,
  'cols': 11,
  'positive_rate': 0.32244988499121185}}

## Step 2 — Train baseline model (v1)

Trains a `GradientBoostingClassifier` on `synthetic_v1.pkl` only.
This is the v1 baseline — no ALNS states are required yet
(`require_alns_states=False`).

**What this cell does:**
- Prints seed provenance for the train/test split and model `random_state`.
- Runs integrity checks on the dataset (feature version, NaN, label values).
- Prints a dataset summary: rows, positive rate, per-source counts.
- Trains with class-balanced sample weights and early stopping.
- Evaluates ROC-AUC + Average Precision on a 15% holdout.
- Checks quality gates and warns if scores are below the thresholds.
- Saves `repair_model_v1.pkl` with the model, scaler, metrics, and seed provenance.

**Quality thresholds**: ROC-AUC ≥ 0.80, Average Precision ≥ 0.60.


In [ ]:
train_repair_model(
    TrainRepairModelConfig(
        data=[str(DATA_DIR / "synthetic_v1.pkl")],
        output=str(TRAINING_DIR / "repair_model_v1.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=False,   # v1 baseline — ALNS data not yet available
        cv_folds=5,
        no_learning_curves=True,
        no_plots=True,
    )
)


REPRODUCIBILITY — SEEDS
  config.seed          : 42  (train/test split + model)
  Python random seed   : set to 42
  numpy random seed    : set to 42

PHASE 1: LOADING DATASET(S)

────────────────────────────────────────────────────────────
DATASET INTEGRITY CHECKS
────────────────────────────────────────────────────────────
  ✓ synthetic_v1.pkl
      source       : synthetic
      rows         : 707,772
      feature_ver  : 2
      pos_rate     : 0.3224
      NaN/inf      : none
      labels       : [0, 1]

  Merged totals
    rows         : 707,772
    positive rate: 0.3224
    ALNS rows    : 0  (0.0% of total)
    [synthetic] : 707,772 rows
────────────────────────────────────────────────────────────

Merged dataset : 707,772 rows x 11 features
  Positive rate : 0.3224
  Class 0 (neg) : 479,551
  Class 1 (pos) : 228,221
  ALNS ratio    : 0.0%

PHASE 2: TRAIN/TEST SPLIT
  random_state = 42
Training set : 601,606 samples
Test set     : 106,166 samples

PHASE 3: MODEL TRAINING
Training

{'model': GradientBoostingClassifier(learning_rate=0.03, max_depth=6, min_samples_leaf=10,
                            min_samples_split=20, n_estimators=500,
                            n_iter_no_change=50, random_state=42,
                            subsample=0.75),
 'scaler': StandardScaler(),
 'feature_version': 2,
 'n_features': 11,
 'metrics': {'accuracy': 0.8383380743364166,
  'precision': 0.752529735487307,
  'recall': 0.742967312242573,
  'f1': 0.7477179521100674,
  'roc_auc': 0.905329090051007,
  'average_precision': 0.86082190830996,
  'confusion_matrix': {'true_negatives': 63569,
   'false_positives': 8364,
   'false_negatives': 8799,
   'true_positives': 25434},
  'specificity': 0.8837251331099774,
  'sensitivity': 0.742967312242573,
  'optimal_threshold': 0.47247143772125455},
 'cv_scores': [0.90545978551478,
  0.9036367633914447,
  0.9047749119953599,
  0.9046839659511408,
  0.9052260363890505],
 'seed': 42,
 'dataset_summary': {'rows': 707772,
  'cols': 11,
  'positive

## Step 3 — Collect ALNS repair states (covariate-shift mitigation)

The baseline model is trained on BFD-labelled FFD states. During actual ALNS
search, the solver visits states that BFD never produces — partially destroyed
solutions with arbitrary residual bin loads. This **covariate shift** can degrade
repair quality.

`collect_alns_states` runs the v1 model on 500 fresh instances, captures every
repair state it encounters, and re-labels those states with the BFD oracle.
This is a single-pass **DAgger-lite** approach: cheaper than true DAgger, but
already substantially reduces the distribution gap.

**Expected output**: row count and positive rate for the captured states;
the file is tagged `source='alns_states'` so the training script can reliably
count and report the ALNS proportion.


In [ ]:
collect_alns_states(
    CollectAlnsStatesConfig(
        model_path=str(TRAINING_DIR / "repair_model_v1.pkl"),
        instances=500,
        n_min=50,
        n_max=200,
        max_negatives=5,
        iterations=200,
        seed=ALNS_SEED,
        output=str(DATA_DIR / "alns_states_v1.pkl"),
    )
)


Seed          : 1
Running ALNS on 500 instances to collect repair states...


Collected 314,713 rows  (pos_rate=0.291)
Saved to: /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/alns_states_v1.pkl  (14.4 MB)
Next step: retrain with additional training data from /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/alns_states_v1.pkl


{'X': array([[0.33688945, 0.1134945 , 0.7152778 , ..., 0.6604305 , 0.6604305 ,
         0.9921075 ],
        [0.33688945, 0.1134945 , 0.7152778 , ..., 0.37981996, 0.21192408,
         0.82519174],
        [0.41473198, 0.17200261, 0.5694444 , ..., 0.58334017, 0.58334017,
         0.99537313],
        ...,
        [0.27907473, 0.0778827 , 0.852459  , ..., 0.5083138 , 0.5083138 ,
         0.567587  ],
        [0.27907473, 0.0778827 , 0.852459  , ..., 0.5862979 , 0.5862979 ,
         0.67457896],
        [0.27907473, 0.0778827 , 0.852459  , ..., 0.5728843 , 0.5728843 ,
         0.6533938 ]], dtype=float32),
 'y': array([1, 0, 1, ..., 0, 0, 0], dtype=int32),
 'feature_version': 2,
 'source': 'alns_states',
 'summary': {'instances': 500,
  'iterations': 200,
  'seed': 1,
  'rows': 314713,
  'positive_rate': 0.2914719125044088}}

## Step 4 — Generate supplementary synthetic data and retrain (v2)

Generates a smaller supplementary synthetic dataset (`synthetic_v2.pkl`, 2 000
instances) and retrains by merging it with the ALNS states collected in Step 3.

The v2 training enforces `require_alns_states=True`: it raises immediately if
no ALNS-tagged dataset is present, making the covariate-shift requirement
explicit and impossible to accidentally skip.

**Expected output**:
- Dataset summary with the ALNS ratio (ALNS rows / total rows) printed.
- Covariate-shift gate: `✓ ALNS states found: N rows (X.X% of total)`.
- Quality gate results for ROC-AUC and Average Precision.
- Model saved as `repair_model_v2.pkl` with the full provenance bundle.


In [ ]:
# 4a. Generate the supplementary synthetic dataset
generate_dataset(
    GenerateDatasetConfig(
        instances=2000,
        n_min=50,
        n_max=200,
        max_negatives=3,
        seed=SYNTHETIC_V2_SEED,
        workers=1,
        output=str(DATA_DIR / "synthetic_v2.pkl"),
    )
)

# 4b. Retrain with synthetic + ALNS-state data
train_repair_model(
    TrainRepairModelConfig(
        data=[
            str(DATA_DIR / "synthetic_v2.pkl"),
            str(DATA_DIR / "alns_states_v1.pkl"),
        ],
        output=str(TRAINING_DIR / "repair_model_v2.pkl"),
        seed=SEED,
        min_roc_auc=0.80,
        min_average_precision=0.60,
        require_alns_states=True,   # v2: raises if no ALNS data detected
        cv_folds=3,
        no_learning_curves=True,
        no_plots=True,
    )
)


DATASET GENERATION
  Instances          : 2000
  Instance size range: [50, 200] items
  Max negatives      : 3
  Seed               : 2
  Workers            : 1
  Output             : /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v2.pkl


Generating dataset: 100%|██████████| 2000/2000 [00:04<00:00, 460.48it/s]



Dataset: 275,652 rows x 11 features
  Positive rate : 0.4117  (expected min: 0.2500)
  Class 0 (neg) : 162,154
  Class 1 (pos) : 113,498

Saved: /content/bin-packing-optimization/bin_packing_optimization/hybrid_learning_metaheuristics/hybrid_alns/repair_model_training/training_data/synthetic_v2.pkl  (12.6 MB)
Next step: pass this file to train_repair_model.py via the data config field
REPRODUCIBILITY — SEEDS
  config.seed          : 42  (train/test split + model)
  Python random seed   : set to 42
  numpy random seed    : set to 42

PHASE 1: LOADING DATASET(S)

────────────────────────────────────────────────────────────
DATASET INTEGRITY CHECKS
────────────────────────────────────────────────────────────
  ✓ synthetic_v2.pkl
      source       : synthetic
      rows         : 275,652
      feature_ver  : 2
      pos_rate     : 0.4117
      NaN/inf      : none
      labels       : [0, 1]
  ✓ alns_states_v1.pkl
      source       : alns_states
      rows         : 314,713
      feature

{'model': GradientBoostingClassifier(learning_rate=0.03, max_depth=6, min_samples_leaf=10,
                            min_samples_split=20, n_estimators=500,
                            n_iter_no_change=50, random_state=42,
                            subsample=0.75),
 'scaler': StandardScaler(),
 'feature_version': 2,
 'n_features': 11,
 'metrics': {'accuracy': 0.854835977641014,
  'precision': 0.7768615460638068,
  'recall': 0.8171127858627859,
  'f1': 0.7964789512847712,
  'roc_auc': 0.931284002269553,
  'average_precision': 0.8982848675321816,
  'confusion_matrix': {'true_negatives': 50546,
   'false_positives': 7225,
   'false_negatives': 5630,
   'true_positives': 25154},
  'specificity': 0.8749372522545914,
  'sensitivity': 0.8171127858627859,
  'optimal_threshold': 0.4763279604533214},
 'cv_scores': [0.9305106557105142, 0.9302156035747075, 0.9305314475623468],
 'seed': 42,
 'dataset_summary': {'rows': 590365,
  'cols': 11,
  'positive_rate': 0.3476290091722917,
  'alns_rows': 

## Step 5 — Optional benchmark

Run the Falkenauer-U benchmark to measure end-to-end solver quality with the
newly trained v2 model.  Set `RUN_BENCHMARK = True` to execute.

This step is intentionally disabled by default because it can take 10–30 minutes
depending on the runtime and instance size.


In [ ]:
RUN_BENCHMARK = False

if RUN_BENCHMARK:
    from bin_packing_optimization.hybrid_learning_metaheuristics.hybrid_alns import (
        hybrid_alns_solver,
    )
    from bin_packing_optimization.utilities.benchmarking import create_benchmark

    benchmark = create_benchmark(
        dataset_key="falkenauer-u",
        solver_module=hybrid_alns_solver,
        time_limit=None,
    )
    benchmark.run(method=None, method_args={"max_iterations": 500})
    csv_path = benchmark.save_results_to_csv()
    print("Results saved to:", csv_path)

## Validation checklist

After running all cells, verify the following in the printed output:

- [ ] **Reproducibility**: seed constants printed at Step 3 setup match the values defined above
- [ ] **Integrity**: all dataset files pass `✓` checks (feature version, NaN, labels)
- [ ] **Dataset summary**: rows, positive rate, per-source counts printed for each training run
- [ ] **ALNS ratio**: v2 training reports `ALNS rows: N (X.X% of total)`
- [ ] **Covariate-shift gate**: `require_alns_states=True` accepted without error
- [ ] **Quality gates**: `✓ ROC-AUC ≥ 0.80` and `✓ Average Precision ≥ 0.60` for both models
- [ ] **Model saved**: `repair_model_v1.pkl` and `repair_model_v2.pkl` present in `TRAINING_DIR`
